In [ ]:
import os, json
import numpy as np
from scipy.stats import ttest_rel, wilcoxon


def load_all_examples_with_prefix(files_with_prefix):
    """
    files_with_prefix: list of (prefix, jsonl_path)
    Returns {new_id: obj} where new_id = f"{prefix}_{old_id}"
    """
    all_examples = {}

    for prefix, path in files_with_prefix:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                old_id = obj.get("id")
                new_id = f"{prefix}_{old_id}"
                obj["id"] = new_id
                all_examples[new_id] = obj

    return all_examples


def collect_latest_runs_for_subdirs(model_dir, subdirs):
    """
    model_dir: .../BASE_JUDGE/<model>
    subdirs: e.g. ["neutral_1","neutral_2","neutral_3"]

    Returns:
      ua_list: [(subdir, ua_results.jsonl), ...]
      ru_list: [(subdir, ru_results.jsonl), ...]
    """
    def find_latest_run(sd_path, prefix):
        runs = [
            d for d in os.listdir(sd_path)
            if d.startswith(prefix) and os.path.isdir(os.path.join(sd_path, d))
        ]
        if not runs:
            return None
        runs.sort()
        latest = runs[-1]
        fname = "ua_results.jsonl" if prefix.startswith("ua") else "ru_results.jsonl"
        fpath = os.path.join(sd_path, latest, fname)
        return fpath if os.path.isfile(fpath) else None

    ua_files, ru_files = [], []

    for sd in subdirs:
        sd_path = os.path.join(model_dir, sd)
        if not os.path.isdir(sd_path):
            continue

        ua_path = find_latest_run(sd_path, "ua_run_")
        ru_path = find_latest_run(sd_path, "ru_run_")

        if ua_path:
            ua_files.append((sd, ua_path))
        if ru_path:
            ru_files.append((sd, ru_path))

    return ua_files, ru_files


# -------------------------
# Stats utils (UA vs RU)
# -------------------------
def scores_from_examples(examples: dict, score_key: str) -> dict:
    """
    examples: {id: obj}
    returns: {id: score}
    """
    out = {}
    for ex_id, obj in examples.items():
        if score_key in obj and obj[score_key] is not None:
            out[ex_id] = float(obj[score_key])
    return out


def ua_vs_ru_significance_from_examples(
    ua_examples: dict,
    ru_examples: dict,
    ua_key="score_ua_mt_vs_ua_human",
    ru_key="score_ru_mt_vs_ru_human",
):
    ua_scores = scores_from_examples(ua_examples, ua_key)
    ru_scores = scores_from_examples(ru_examples, ru_key)

    common = sorted(set(ua_scores) & set(ru_scores))
    if len(common) == 0:
        raise ValueError("No overlapping ids between UA and RU examples dicts.")

    x = np.array([ua_scores[i] for i in common], dtype=float)
    y = np.array([ru_scores[i] for i in common], dtype=float)

    diff = x - y

    res = {
        "n_ua": len(ua_scores),
        "n_ru": len(ru_scores),
        "n_common": len(common),
        "mean_UA": float(x.mean()),
        "mean_RU": float(y.mean()),
        "mean_diff_UA_minus_RU": float(diff.mean()),
        "median_diff_UA_minus_RU": float(np.median(diff)),
    }

    t = ttest_rel(x, y)
    res["paired_ttest"] = {"t": float(t.statistic), "p": float(t.pvalue)}

    try:
        w = wilcoxon(x, y, zero_method="wilcox")
        res["wilcoxon"] = {"W": float(w.statistic), "p": float(w.pvalue)}
    except ValueError as e:
        res["wilcoxon"] = {"error": str(e)}

    return res


# -------------------------
# Direction decisions + kappa between judges
# -------------------------
def direction_labels_from_examples(
    ua_examples: dict,
    ru_examples: dict,
    ua_key="score_ua_mt_vs_ua_human",
    ru_key="score_ru_mt_vs_ru_human",
    eps=0.0,
):
    """
    Returns:
      labels: {id: "UA"|"RU"|"TIE"} where UA means UA distance < RU distance by eps
      diffs:  {id: (ua_score - ru_score)}
    """
    ua_scores = scores_from_examples(ua_examples, ua_key)
    ru_scores = scores_from_examples(ru_examples, ru_key)

    common = sorted(set(ua_scores) & set(ru_scores))
    labels, diffs = {}, {}

    for i in common:
        d = float(ua_scores[i] - ru_scores[i])
        diffs[i] = d
        if d < -eps:
            labels[i] = "UA"
        elif d > eps:
            labels[i] = "RU"
        else:
            labels[i] = "TIE"

    return labels, diffs


def cohen_kappa_from_label_dicts(labels_a: dict, labels_b: dict):
    """
    Cohen's kappa on common ids, chance-corrected agreement.
    """
    common = sorted(set(labels_a) & set(labels_b))
    if not common:
        raise ValueError("No overlapping ids between the two label dicts.")

    a = [labels_a[i] for i in common]
    b = [labels_b[i] for i in common]

    cats = sorted(set(a) | set(b))
    cat2idx = {c: j for j, c in enumerate(cats)}
    n = len(common)

    # confusion counts
    M = np.zeros((len(cats), len(cats)), dtype=float)
    for x, y in zip(a, b):
        M[cat2idx[x], cat2idx[y]] += 1.0

    po = float(np.trace(M) / n)  # observed agreement
    pa = M.sum(axis=1) / n       # marginals
    pb = M.sum(axis=0) / n
    pe = float(np.sum(pa * pb))  # expected by chance

    kappa = (po - pe) / (1.0 - pe) if (1.0 - pe) > 1e-12 else 0.0
    return {
        "n_common": n,
        "categories": cats,
        "pct_agreement": po,
        "kappa": float(kappa),
        "confusion": M.tolist(),
    }


def print_res(title: str, res: dict):
    print(f"\n--- {title} ---")
    for k, v in res.items():
        if isinstance(v, dict):
            print(f"{k}: {json.dumps(v, indent=2)}")
        else:
            print(f"{k}: {v}")



In [ ]:
def direction_labels_binary_from_examples(
    ua_examples: dict,
    ru_examples: dict,
    ua_key="score_ua_mt_vs_ua_human",
    ru_key="score_ru_mt_vs_ru_human",
    tie_break="RU",  # what to do if exactly equal, float, so chances are minimal
):
    """
    Returns {id: "UA"|"RU"} (no TIE).
    Rule:
      - UA if ua_score < ru_score
      - RU if ua_score > ru_score
      - if equal: tie_break
    """
    ua_scores = scores_from_examples(ua_examples, ua_key)
    ru_scores = scores_from_examples(ru_examples, ru_key)

    common = sorted(set(ua_scores) & set(ru_scores))
    labels = {}

    for i in common:
        ua = float(ua_scores[i])
        ru = float(ru_scores[i])
        if ua < ru:
            labels[i] = "UA"
        elif ru < ua:
            labels[i] = "RU"
        else:
            labels[i] = tie_break  # exact equality

    return labels


def cohen_kappa_binary(labels_a: dict, labels_b: dict):
    common = sorted(set(labels_a) & set(labels_b))
    if not common:
        raise ValueError("No overlapping ids between the two label dicts.")

    a = [labels_a[i] for i in common]
    b = [labels_b[i] for i in common]
    print(a)
    print(b)

    # categories fixed for binary
    cats = ["RU", "UA"]
    cat2idx = {c: j for j, c in enumerate(cats)}
    n = len(common)

    M = np.zeros((2, 2), dtype=float)
    for x, y in zip(a, b):
        M[cat2idx[x], cat2idx[y]] += 1.0

    po = float(np.trace(M) / n)
    pa = M.sum(axis=1) / n
    pb = M.sum(axis=0) / n
    pe = float(np.sum(pa * pb))
    kappa = (po - pe) / (1.0 - pe) if (1.0 - pe) > 1e-12 else 0.0

    return {
        "n_common": n,
        "pct_agreement": po,
        "kappa": float(kappa),
        "confusion_RU_UA": M.tolist(),  # rows=judgeA, cols=judgeB
    }
